# 使用双端口网络分析仪测量多端口设备

## 简介

在微波测量中，通常需要使用 m 端口网络分析仪（显然 $m<n$）来测量 n 端口设备。

<img src="nports_with_2ports.svg"/>

这可以通过在每个未测量的端口端接匹配负载，并假设反射功率可以忽略来实现。通过多次测量，然后可以重构原始的 n 端口。此示例的第一部分演示了此方法。

然而，在某些情况下，这可能无法提供最准确的结果，甚至在所有测量环境中都不可能。或者，有时无法为所有端口提供匹配负载。此示例的第二部分提出了使用阻抗重新归一化的优雅解决方案。我们称之为 *Tippet's technique*，因为它听起来很好。

</cell id="cell-2">

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.network import connect, subnetwork

%matplotlib inline

rf.stylely()

## 匹配端口

In [ ]:
tee = rf.data.tee
print(tee)

为了演示目的，我们将通过提取原始网络的 3 个子集，即 3 个子网络来"伪造"这 3 个不同的测量：

In [ ]:
# 2 port Networks as if one measures the tee with a 2 ports VNA
tee12 = subnetwork(tee, [0, 1])  # 2 port Network btw ports 1 & 2, port 3 being matched
tee23 = subnetwork(tee, [1, 2])  # 2 port Network btw ports 2 & 3, port 1 being matched
tee13 = subnetwork(tee, [0, 2])  # 2 port Network btw ports 1 & 3, port 2 being matched

实际上，这 3 个网络来自使用不同端口对进行的 3 次测量，未使用的端口已正确匹配。

在使用 `n_twoports_2_nport` 函数之前，必须通过设置 `Network.name` 属性来定义这些子集的名称，以便函数知道哪个对应于什么：

In [ ]:
tee12.name = 'tee12'
tee23.name = 'tee23'
tee13.name = 'tee13'

现在我们可以从这 3 个 2 端口子网络构建 3 端口网络：

In [ ]:
ntw_list = [tee12, tee23, tee13]
tee_rebuilt = rf.network.n_twoports_2_nport(ntw_list, nports=3)
print(tee_rebuilt)

In [ ]:
# this is an ideal example, both Network are thus identical
print(tee == tee_rebuilt)

## Tippet's 技术

## Tippet's 技术概述

按照 [1] 中给出的示例，使用双端口网络分析仪测量 4 端口网络。

该技术的概述：

1. 校准双端口网络分析仪
2. 获取四个已知端接（$Z_1, Z_2, Z_3,Z_4$）。其中最多一个可以有 $|\Gamma| = 1$
3. 测量所有 2 端口子网络的组合（共有 6 个）。每个当前未测量的端口必须用其对应的负载端接。
4. 将每个子网络重新归一化到在不被测量时用于端接它的负载阻抗。
5. 构建复合 4 端口，重新归一化为 VNA 阻抗。

## 实现

首先，我们创建一个 Media 对象，用于生成测试网络。我们将使用 WR-10 矩形波导。

In [ ]:
wg = rf.instances.wr10

接下来，让我们生成一个随机的 4 端口网络，它将是我们试图使用双端口网络分析仪测量的 DUT。

In [ ]:
dut = wg.random(n_ports  = 4,name= 'dut')
dut

现在，我们需要定义用于在未测量端口时端接的负载，如 [1] 中所述，最多只能有一个具有完全反射 $|\Gamma| = 1$

In [ ]:
loads = [wg.load(.1+.1j),
         wg.load(.2-.2j),
         wg.load(.3+.3j),
         wg.load(.5),
         ]
# construct the impedance array, of shape FXN
z_loads = np.array([k.z.flatten() for k in loads]).T


创建所需的测量端口组合。测量一个 4 端口需要 6 个不同的测量，使用双端口 VNA。一般来说，#measurements = $n\choose 2$，对于 2 端口 VNA 上的 n 端口 DUT。

In [ ]:
ports = np.arange(dut.nports)
port_combos = list(combinations(ports, 2))
port_combos

现在开始做。好的，我们遍历端口组合并将负载连接到正确的位置，模拟实际测量。每个原始子网络测量都被保存，以及重新归一化的子网络。最后，我们将结果放入 4 端口复合网络。

In [ ]:
composite = wg.match(nports = 4)  # composite network, to be filled.
measured,measured_renorm = {},{}  # measured subnetworks and renormalized sub-networks

# ports  `a` and `b` are the ports we will connect the VNA too
for a,b in port_combos:
    # port `c` and `d` are the ports which we will connect the loads too
    c,d =ports[(ports!=a)& (ports!=b)]

    # determine where `d` will be on four_port, after its reduced to a three_port
    e = np.where(ports[ports!=c]==d)[0][0]

    # connect loads
    three_port = connect(dut,c, loads[c],0)
    two_port =  connect(three_port,e, loads[d],0)

    # save raw and renormalized 2-port subnetworks
    measured['%i%i'%(a,b)] = two_port.copy()
    two_port.renormalize(np.c_[z_loads[:,a],z_loads[:,b]])
    measured_renorm['%i%i'%(a,b)] = two_port.copy()

    # stuff this 2-port into the composite 4-port
    for i,m in enumerate([a,b]):
        for j,n in enumerate([a,b]):
            composite.s[:,m,n] = two_port.s[:,i,j]

    # properly copy the port impedances
    composite.z0[:,a] = two_port.z0[:,0]
    composite.z0[:,b] = two_port.z0[:,1]

# finally renormalize from load z0 to 50 ohms
composite.renormalize(50)

## 结果

### 自洽性

请注意，6 次测量 2 端口子网络总共产生 24 个 s 参数，但我们只需要 16 个。这是因为每个反射 s 参数被测量了 3 次。如 [1] 中所述，我们将使用这种冗余测量作为准确性的检查。

重新归一化的网络存储在一个字典中，名称基于它们的端口索引，从这一点你可以看到每个都已重新归一化为适当的 z0。

In [ ]:
measured_renorm

绘制所有三个 $S_{11}$ 的原始测量，我们可以看到它们不一致。这些图对应 [1] 中的图 5 和图 7

In [ ]:
s11_set = rf.networkSet.NetworkSet([measured[k] for k in measured if k[0]=='0'])

plt.figure(figsize = (8,4))
plt.subplot(121)
s11_set .plot_s_db(0,0)
plt.subplot(122)
s11_set .plot_s_deg(0,0)
plt.tight_layout()

然而，重新归一化的测量完美一致。这些图对应 [1] 中的图 6 和图 8

In [ ]:
s11_set = rf.networkSet.NetworkSet([measured_renorm[k] for k in measured_renorm if k[0]=='0'])

plt.figure(figsize = (8,4))
plt.subplot(121)
s11_set .plot_s_db(0,0)
plt.subplot(122)
s11_set .plot_s_deg(0,0)
plt.tight_layout()

### 准确性测试

确保我们的复合网络与 DUT 相同

In [ ]:
composite == dut

不错！有多接近？

In [ ]:
sum((composite - dut).s_mag)

糟糕！

## 实际应用

这可以用于许多方式。在波导中，你可以在标准的双端口校准（如 TRL）后直接测量辐射开路。然后使用 *Tippet's technique*，你可以在不测量时将每个端口保持开路。这样你就不必购买一堆负载。那该多好啊？

## 更复杂的模拟

In [ ]:
def tippits(dut, gamma, noise=None):
    """simulate tippits technique on a 4-port dut.
    """
    ports = np.arange(dut.nports)
    port_combos = list(combinations(ports, 2))

    loads = [wg.load(gamma) for k in ports]

    # construct the impedance array, of shape FXN
    z_loads = np.array([k.z.flatten() for k in loads]).T
    composite = wg.match(nports = dut.nports)  # composite network, to be filled.

    # ports  `a` and `b` are the ports we will connect the VNA too
    for a,b in port_combos:
        # port `c` and `d` are the ports which we will connect the loads too
        c,d =ports[(ports!=a)& (ports!=b)]

        # determine where `d` will be on four_port, after its reduced to a three_port
        e = np.where(ports[ports!=c]==d)[0][0]

        # connect loads
        three_port = connect(dut,c, loads[c],0)
        two_port =  connect(three_port,e, loads[d],0)

        if noise is not None:
            two_port.add_noise_polar(*noise)
        # save raw and renormalized 2-port subnetworks
        measured['%i%i'%(a,b)] = two_port.copy()
        two_port.renormalize(np.c_[z_loads[:,a],z_loads[:,b]])
        measured_renorm['%i%i'%(a,b)] = two_port.copy()

        # stuff this 2-port into the composite 4-port
        for i,m in enumerate([a,b]):
            for j,n in enumerate([a,b]):
                composite.s[:,m,n] = two_port.s[:,i,j]

        # properly copy the port impedances
        composite.z0[:,a] = two_port.z0[:,0]
        composite.z0[:,b] = two_port.z0[:,1]

    # finally renormalize from load z0 to 50 ohms
    composite.renormalize(50)

    return composite

In [ ]:
dut = wg.random(4)

def er(gamma, *args):
    return max(abs(tippits(dut, rf.mathFunctions.db_2_mag(gamma),*args).s_db-dut.s_db).flatten())

gammas = np.linspace(-40,-0.1,11)

plt.title(r'Error vs $|\Gamma|$')
plt.plot(gammas, [er(k) for k in gammas])
plt.semilogy()
plt.xlabel(r'$|\Gamma|$ of Loads (dB)')
plt.ylabel('Max Error in DUT\'s dB(S)')

plt.figure()
noise = (1e-5,.1)
plt.title(r'Error vs $|\Gamma|$ with reasonable noise')
plt.plot(gammas, [er(k, noise) for k in gammas])
plt.semilogy()
plt.xlabel(r'$|\Gamma|$ of Loads (dB)')

plt.ylabel('Max Error in DUT\'s dB(S)')